In [1]:
from utils.orthanc_utils import *
from utils.db_utils import *
from utils.plot_utils import *
from utils.pipeline import *


def get_volumes(dicoms):    
    df = pd.DataFrame(
        [{"SliceLocation": d.SliceLocation, "InstanceNumber": d.InstanceNumber, "PixelArray": d.pixel_array} for d in dicoms]
    ).sort_values(["SliceLocation", "InstanceNumber"])

    d0 = dicoms[0]
    pixel_spacing = float(d0.PixelSpacing[0])
    thickness = float(getattr(d0, "SpacingBetweenSlices", getattr(d0, "SliceThickness", np.nan)))
    voxel_size = (pixel_spacing ** 2 * thickness) / 1000
    
    array_4d = []
    for _, slice_df in df.groupby("SliceLocation"):
        time_array = []
        for _, time_df in slice_df.groupby("InstanceNumber"):
            pixel_array = time_df["PixelArray"].iloc[0]
            time_array.append(pixel_array)

        slice_array = np.stack(time_array, axis=-1)
        array_4d.append(slice_array)

    array_4d = np.stack(array_4d, axis=-2)

    labels = {500: "lv", 1000: "rv", 1500: "lv_myo", 2000: "rv_myo"}
    curves = {
        name: np.sum(array_4d == val, axis=(0, 1, 2)) * voxel_size
        for val, name in labels.items()
    }
    lv, rv = curves["lv"], curves["rv"]
    lv_edv, lv_esv = lv.max(), lv.min()
    rv_edv, rv_esv = rv.max(), rv.min()

    metrics = {
        "lv_edv": lv_edv,
        "lv_esv": lv_esv,
        "lv_ef": (lv_edv - lv_esv) / lv_edv,
        "lv_mass": np.mean(curves["lv_myo"]) * 1.05,
        "rv_edv": rv_edv,
        "rv_esv": rv_esv,
        "rv_ef": (rv_edv - rv_esv) / rv_edv,
        "rv_mass": np.mean(curves["rv_myo"]) * 1.05,
    }

    return metrics


/home/tina/Documents/PhD/Project/imageCLASP/clasp/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
db = TinyDB(DB_PATH)
studies = fetch_db_studies()

study = studies[0]

In [3]:
df = pd.DataFrame([series.__dict__ for series in study.series_dict.values()])
sax_dl_df = df[(df['dl_orthanc_id'].notna()) & (df['roundel_orthanc_id'].isna())]

for series_id, dl_series_id in zip(sax_dl_df["orthanc_series_id"], sax_dl_df["dl_orthanc_id"]):
    series = study.series_dict[series_id]
    image_dicoms = fetch_orthanc_dicoms_for_series(series_id)
    mask_dicoms = fetch_orthanc_dicoms_for_series(dl_series_id)

    masked_images = [image.pixel_array * (mask.pixel_array > 0) for image, mask in zip(image_dicoms, mask_dicoms) ]
    new_orthanc_id = send_series_to_orthanc(masked_images, image_dicoms, new_description='Roundel Processed')
    series.roundel_orthanc_id = new_orthanc_id

metrics = get_volumes(fetch_orthanc_dicoms_for_series_list(df["dl_orthanc_id"].dropna().unique()))
metrics['orthanc_study_id'] = study.orthanc_study_id
metrics['patient_id'] = study.patient_id
metrics['study_date'] = study.study_date

metrics_df = pd.read_csv(METRICS_PATH)
metrics_df = pd.concat([metrics_df, pd.DataFrame([metrics])]).set_index('orthanc_study_id')
metrics_df.to_csv(METRICS_PATH)


/tmp/ipykernel_203371/540095361.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_df = pd.concat([metrics_df, pd.DataFrame([metrics])]).set_index('orthanc_study_id')


In [4]:
metrics_df

,patient_id,study_date,lv_edv,lv_esv,lv_ef,lv_mass,rv_edv,rv_esv,rv_ef,rv_mass
orthanc_study_id,,,,,,,,,,
c84ff965-9b667fef-54070af3-b1386f1a-42841c76,67812345,20251201,30.866044,13.072321,0.576482,23.216613,58.395765,32.574646,0.442175,11.112382


In [2]:
import tk